# Content-Based Recommender System

Recommender systems are algorithms which suggest information of interest to the user. If you ever used the internet, you will surely have made contact with a recommender system. Netflix, Amazon, Spotify, all make use of such functions to improve user satisfaction as well as buying behaviour. Based on your previous activity the recommender system will suggest you the next video, product, song or whatever. The most common recommender types are content-based, collaborative filtering and hybrid models of those two types.

Having enough user data do build a proper collaborative model is usually an obstacle at the beginning, as there is little known about the user's behaviour. This is also referred to as the cold start problem in recommender systems. In this notebook you will learn how to build a content-based recommender-system. A content-based model has the advantage that no user data is required and the model can be scaled to a large number of users independently of their preferences. Solving the cold start problem. For a content-based recommender system the filtering is based on the item features only.
<br/><br/>
<p align="center">
    <img src="images/book_text.png" alt="drawing" width="400"/>
</p>
<br/><br/>
An example: If you have a book that has the features "aquarium" and "fish" you would get recommendations with the same features, so books about aquarium fish without any usage of ratings or user data.


## Learning Objectives

At the end of this notebook, you should be able to:

- Build a content-based recommender that ranks items by the cosine similarity of their features.
- Compute cosine similarity between one-hot encoded item vectors.
- Apply TF-IDF to turn item descriptions into vectors for a text-based recommender.
- Generate the top-N most similar items for a given item.

Let's start with building your own content-based recommender system!

Import packages:

In [1]:
# requirements
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.metrics.pairwise import cosine_similarity

## Cosine similarity 

To get meaningful recommendations, it is important to **measure** the **similarity** of two items. To do so, **cosine similarity** is the most popular metric for content-based models. Mathematically, the **cosine similarity** is the dot product of normalized vectors and **measures** the **cosine of the angle between two vectors** projected in a n-dimensional space (n = number of features). Output values are limited between 0 and 1, where 1 means the vectors are pointing in the same direction and 0 means they are orthogonal.




If you got a two-dimensional space and the angle of two data points is 45° the cosine would be 0.7 while it would be zero for an angle of 90°. In a data frame each feature represents a dimension. This means, every observation (row) could be plotted in a n-dimensional space based on it's feature-properties. Let's have a look at two examples:

### Example 1: 2-dimensional space 

Let's assume, we got two fish with different colors. One fish is solid blue, while the other is solid orange. Plotted in a 2-dimensional space, where one axis represents the color blue and the other the color orange, one fish would have the coordinates (1,0) and the other (0,1). A fish with both colors would be in between (1,1). To calculate the cosine similarity score between two fish, we simply measure the cosine of the two coordinate vectors. 

<br/><br/>
<p align="center">
    <img src="images/fish_text.png" alt="drawing" width="300"/>
</p>
<br/><br/>
The vectors of the blue and orange fish and the solid orange fish would have a 45° angle, therefore a cosine similarity of 0.7. Just the same if we compare the solid blue and the blue-orange fish. The solid blue and the solid orange fish would have an angle of 90° and hence a cosine of 0, so no similarity at all with respect to the feature color. 

### Example 2: 4-dimensional space

Let's create an example data frame:

In [19]:
# create example data frame
d = {
    "green": [1, 1, 0, 1, 1],
    "blue": [1, 0, 0, 1, 1],
    "yellow": [0, 0, 1, 1, 0],
    "black": [1, 0, 0, 1, 1],
}
df_example2 = pd.DataFrame(data=d, index=["fish1", "fish2", "fish3", "fish4", "fish5"])
df_example2

,green,blue,yellow,black
fish1,1,1,0,1
fish2,1,0,0,0
fish3,0,0,1,0
fish4,1,1,1,1
fish5,1,1,0,1


For this example let's think of 5 different fish. Each fish has a different combination of the colors green, blue, yellow and black. Those colors are the features of the fish and likewise the dimensionality of the data. Each feature is represented by a value of 0 or 1, while 0 means the fish doesn't have this color and 1 it has. In our example fish1 has the colors green, blue and yellow. If we assign colors to dimensions we get the coordinates (1,1,0,1). The following equation allows us to calculate the cosine similarity for multi-dimensional data: 

$$
  CosSim(x,y) = \frac{\sum_{i}x_{i}y_{i}}{\sqrt{\sum_{i}x_{i}^2}\sqrt{\sum_{i}y_{i}^2}}
$$

If we calculate the cosine similarity on our data frame, we get:

In [3]:
# calculate cosine similarity
cosine_sim_sk = pd.DataFrame(
    cosine_similarity(df_example2).round(2),
    columns=["fish1", "fish2", "fish3", "fish4", "fish5"],
    index=["fish1", "fish2", "fish3", "fish4", "fish5"],
)
cosine_sim_sk

,fish1,fish2,fish3,fish4,fish5
fish1,1.00,0.58,0.0,0.87,1.00
fish2,0.58,1.00,0.0,0.50,0.58
fish3,0.00,0.00,1.0,0.50,0.00
fish4,0.87,0.50,0.5,1.00,0.87
fish5,1.00,0.58,0.0,0.87,1.00


How to interpret this?

You can see, that fish1 is of course totally similar to itself, indicated by a value of 1. Then fish1 is less similar to fish 2 (only one overlap) as to fish 4 (three overlaps) and has no similarity to fish3 while it is totally similar to fish5.

## Building a feature-based recommender

The first recommender works directly on the fish features: categorical attributes such as origin, diet, and behaviour. To measure similarity we first turn those categories into numbers with one-hot encoding, then compare the resulting vectors with cosine similarity.

Cosine similarity is a sensible default here because it compares the direction of two feature vectors rather than their magnitude. A fish is therefore not judged more similar simply because it has more recorded attributes. An alternative such as Euclidean distance is sensitive to that magnitude, which is why cosine similarity is the more common choice for sparse, high-dimensional feature vectors.

Reading in the data:

In [4]:
# read data
df_fishes = pd.read_csv("data/fish_1.csv")
df_fishes.reset_index(drop=True)
df_fishes

,name,water_parameters,fish_group,compatibility_with_crabs,temperature,pelvic_region,behaviour,compatibility_with_fish,tank_size,breeding,...,origin,diet,vegetation_possible,compatibility_with_snails_shells,compatibility_with_dwarf_crayfish,final_size,level_of_difficulty,characteristics,compatibility_with_large_crayfish,life_expectancy
0,Cardinalfish - Tanichthys albonubes,soft to hard,Beardies,no,18-22 °C,everywhere,active,"yes, with peaceful fish",54 l (ca. 60cm),medium,...,Asia,omnivorous - omnivorous,yes,yes,yes,4-8 cm,1 - easy,dynamic group behaviour,no,3-5 years
1,Red-headed Tetra - Petitella (Hemigrammus) ble...,soft to medium hard,Tetra,NaN,25-30 °C,top,NaN,NaN,100 l (ca. 80cm),hard,...,South America,omnivorous - omnivorous,yes,yes,yes,4-8 cm,2 - normal,NaN,NaN,NaN
2,Blue Neon - Paracheirodon simulans,soft to medium hard,Tetra,NaN,25-30 °C,center,NaN,NaN,54 l (ca. 60cm),hard,...,South America,omnivorous - omnivorous,yes,yes,yes,1-4 cm,1 - easy,NaN,NaN,NaN
3,Red Neon XL - Paracheirodon axelrodi,soft to medium hard,Tetra,NaN,25-30 °C,center,NaN,NaN,54 l (ca. 60cm),hard,...,South America,omnivorous - omnivorous,yes,yes,yes,4-8 cm,1 - easy,NaN,NaN,NaN
4,Neon Tetra - Paracheirodon innesi,soft to medium hard,Tetra,NaN,25-30 °C,top,NaN,NaN,54 l (ca. 60cm),hard,...,South America,omnivorous - omnivorous,yes,yes,yes,1-4 cm,2 - normal,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,Dwarf Snakehead - Channa gachua blue bengal - ...,medium hard water,Other fish,NaN,25-30 °C,below,NaN,NaN,250/300 l (ca. 120cm),medium,...,Asia,carnivore - meat eater,yes,no,no,> 12cm,4 - special,NaN,NaN,NaN
539,Beautiful Snakehead - Channa pulchra - DNZ - 10cm,medium hard water,Other fish,NaN,20-25 °C,below,NaN,NaN,250/300 l (ca. 120cm),NaN,...,Asia,carnivore - meat eater,yes,no,no,> 12cm,4 - special,NaN,NaN,NaN
540,Axolotl - Ambystoma mexicanum - DNZ,medium hard to hard,Other fish,NaN,15-20 °C,below,NaN,NaN,100 l (ca. 80cm),simply,...,Central America,carnivore - meat eater,conditional (see description),conditional (see description),no,> 12cm,4 - special,NaN,NaN,NaN
541,Brown White-fronted Knifefish - Apteronotus al...,soft to hard,Other fish,NaN,25-30 °C,center,NaN,NaN,600 l (ca. 180cm),hard,...,South America,carnivore - meat eater,yes,no,no,> 12cm,4 - special,NaN,NaN,NaN


The dataset contains 543 types of fish, each described by 21 features.

To calculate the cosine similarity of the fish in our data frame, we need to transform the features into binary vectors. To do this we use the get_dummies function implemented in scikit-learn.

In [5]:
# create dummy vectors
df_fishes_vectorized = pd.get_dummies(df_fishes.iloc[:, 1:], drop_first=True).astype(
    int
)
df_fishes_vectorized.head()

,water_parameters_hard water,water_parameters_medium hard to hard,water_parameters_medium hard water,water_parameters_soft to hard,water_parameters_soft to medium hard,water_parameters_soft water,fish_group_Beardies,fish_group_Bone Tongues,fish_group_Carps,fish_group_Catfish,...,characteristics_fish of prey,characteristics_form family groups,characteristics_growth eater,characteristics_interesting color,characteristics_interesting parental care,characteristics_snail eater,compatibility_with_large_crayfish_no,compatibility_with_large_crayfish_yes,life_expectancy_3-5 years,life_expectancy_5-10 years
0,0,0,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,1,0
1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


You can see that this increases the numbers of features from 22 to 93. 

Let's calculate the cosine similarity:

In [6]:
# calculate cosine similarity
cosine_sim_sk = pd.DataFrame(cosine_similarity(df_fishes_vectorized))
cosine_sim_sk

,0,1,2,3,4,5,6,7,8,9,...,533,534,535,536,537,538,539,540,541,542
0,1.000000,0.445399,0.397360,0.381771,0.445399,0.445399,0.500773,0.138343,0.345857,0.397360,...,0.198680,0.132453,0.076472,0.000000,0.000000,0.198680,0.138343,0.000000,0.132453,0.076472
1,0.445399,1.000000,0.720577,0.692308,0.923077,0.923077,0.605406,0.501745,0.334497,0.480384,...,0.400320,0.240192,0.277350,0.000000,0.000000,0.160128,0.083624,0.000000,0.240192,0.277350
2,0.397360,0.720577,1.000000,0.880705,0.800641,0.800641,0.700140,0.348155,0.261116,0.666667,...,0.500000,0.333333,0.192450,0.102062,0.102062,0.166667,0.087039,0.102062,0.333333,0.192450
3,0.381771,0.692308,0.880705,1.000000,0.692308,0.692308,0.605406,0.334497,0.418121,0.640513,...,0.480384,0.320256,0.184900,0.098058,0.098058,0.160128,0.083624,0.098058,0.320256,0.184900
4,0.445399,0.923077,0.800641,0.692308,1.000000,1.000000,0.672673,0.501745,0.334497,0.560449,...,0.400320,0.240192,0.277350,0.000000,0.000000,0.160128,0.083624,0.000000,0.240192,0.277350
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,0.198680,0.160128,0.166667,0.160128,0.160128,0.160128,0.070014,0.435194,0.174078,0.250000,...,0.500000,0.583333,0.384900,0.408248,0.408248,1.000000,0.870388,0.408248,0.583333,0.384900
539,0.138343,0.083624,0.087039,0.083624,0.083624,0.083624,0.073127,0.545455,0.181818,0.174078,...,0.435194,0.522233,0.502519,0.426401,0.426401,0.870388,1.000000,0.426401,0.522233,0.502519
540,0.000000,0.000000,0.102062,0.098058,0.000000,0.000000,0.085749,0.213201,0.106600,0.102062,...,0.306186,0.408248,0.353553,1.000000,1.000000,0.408248,0.426401,1.000000,0.408248,0.353553
541,0.132453,0.240192,0.333333,0.320256,0.240192,0.240192,0.140028,0.435194,0.087039,0.250000,...,0.416667,1.000000,0.481125,0.408248,0.408248,0.583333,0.522233,0.408248,1.000000,0.481125


In the above similarity matrix, the index and columns represent the fishes, and the corresponding values represent the cosine similarity. You can see that fish1 is totally similar to itself, indicated by a value of 1. Then fish1 is less similar to fish2 (only one overlap) than to fish4 (three overlaps), has no similarity to fish3, and is totally similar to fish5.

And finally the function to get recommendations!

Now that we have the cosine similarities, we can simply sort the similarities with respect to a specific item. This function takes in the name of a fish, checks the similarity scores compared to other fish, sorts the values and gives out the 10 most similar fish based on similarity score. Just type in a name of a fish in the data frame and you will get recommended the 10 most similar fish.

In [7]:
# Build index with fish names
names = df_fishes["name"]
indices = pd.Series(df_fishes.index, index=df_fishes["name"])

In [8]:
# Name of the fishes
names

0                    Cardinalfish - Tanichthys albonubes
1      Red-headed Tetra - Petitella (Hemigrammus) ble...
2                     Blue Neon - Paracheirodon simulans
3                   Red Neon XL - Paracheirodon axelrodi
4                      Neon Tetra - Paracheirodon innesi
                             ...                        
538    Dwarf Snakehead - Channa gachua blue bengal - ...
539    Beautiful Snakehead - Channa pulchra - DNZ - 10cm
540                  Axolotl - Ambystoma mexicanum - DNZ
541    Brown White-fronted Knifefish - Apteronotus al...
542                Senegal bichir - Polypterus senegalus
Name: name, Length: 543, dtype: str

In [9]:
# Indices of the fishes series
indices

name
Cardinalfish - Tanichthys albonubes                           0
Red-headed Tetra - Petitella (Hemigrammus) bleheri - DNZ      1
Blue Neon - Paracheirodon simulans                            2
Red Neon XL - Paracheirodon axelrodi                          3
Neon Tetra - Paracheirodon innesi                             4
                                                           ... 
Dwarf Snakehead - Channa gachua blue bengal - DNZ 8-10cm    538
Beautiful Snakehead - Channa pulchra - DNZ - 10cm           539
Axolotl - Ambystoma mexicanum - DNZ                         540
Brown White-fronted Knifefish - Apteronotus albifrons       541
Senegal bichir - Polypterus senegalus                       542
Length: 543, dtype: int64

In [10]:
# Function that get fish recommendations based on the cosine similarity
def fish_recommendations(name, top_n=10):
    # get the index of the fish we put into the function
    idx = indices[name]
    # calculate all cosine similarities to that fish and store it in a list
    sim_scores = list(enumerate(cosine_sim_sk[idx]))
    # sort the list staring with the highest similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # get the similarities from 1:11 (not starting with 0 because it is the same fish)
    sim_scores = sim_scores[1 : top_n + 1]
    # get the indices of those 10 fish
    fish_indices = [i[0] for i in sim_scores]
    # return the fish names of that 10 fish
    return names.iloc[fish_indices]

Now, use this function to get recommendations :)

In [11]:
# get recommendations
fish_recommendations("Blue Neon - Paracheirodon simulans", top_n=10)

27     Black Emperor Tetra - Nematobrycon palmeri "bl...
64     Golden Emperor Tetra EU-NZ - Nematobrycon palm...
3                   Red Neon XL - Paracheirodon axelrodi
15                     Copper Tetra DNZ - Hasemania nana
32               Rio Red - Hyphessobrycon flammeus - DNZ
37                         Copper Tetra - Hasemania nana
85                  Dwarf tetra - Nannostomus marginatus
123                       Blue Peru Tetra - Knodus borki
128              Red Neon - Paracheirodon axelrodi - DNZ
29     Black Phantom Tetra - Hyphessobrycon megalopterus
Name: name, dtype: str

### Reading the results

For **Blue Neon - Paracheirodon simulans**, the one-hot recommender returns other small tetras, led by the Black Emperor Tetra, the Golden Emperor Tetra, and the Red Neon XL. The recommendations are plausible but broad: because every categorical feature is weighted equally, fish that merely share common attributes (similar tank size, diet, or behaviour) can rank highly even when they are not close relatives.

## Text related content-based recommender system

Many items, such as products in an online shop or films in a catalogue, come with a free-text description. These descriptions carry information that fixed categorical features can miss, so content-based recommenders are often built on text.

Here we combine every feature of each fish into a single block of text, then convert that text into numbers with TF-IDF. TF-IDF weights each word by how informative it is: words that appear in almost every description count for little, while distinctive words count for more. This is the key difference from the one-hot approach above, where every feature was weighted equally. Alternatives such as a plain word count (`CountVectorizer`) or learned embeddings (`Word2Vec`) are also possible, with TF-IDF offering a simple and strong baseline.

First we have to create a text block, we can work with. Let's join our features to text. 

In [12]:
# Set column types to string for whole data frame
# fillna('') first: in pandas 3.0 astype(str) keeps missing values as NaN
# rather than the literal "nan", so empty cells must be filled before the join
df = df_fishes.fillna("").astype(str).copy()
# Select all columns
columns = df.columns
# Merge features to text block
df["text"] = df[columns].agg(" ".join, axis=1)

Let's have a look at the text.

In [13]:
df

,name,water_parameters,fish_group,compatibility_with_crabs,temperature,pelvic_region,behaviour,compatibility_with_fish,tank_size,breeding,...,diet,vegetation_possible,compatibility_with_snails_shells,compatibility_with_dwarf_crayfish,final_size,level_of_difficulty,characteristics,compatibility_with_large_crayfish,life_expectancy,text
0,Cardinalfish - Tanichthys albonubes,soft to hard,Beardies,no,18-22 °C,everywhere,active,"yes, with peaceful fish",54 l (ca. 60cm),medium,...,omnivorous - omnivorous,yes,yes,yes,4-8 cm,1 - easy,dynamic group behaviour,no,3-5 years,Cardinalfish - Tanichthys albonubes soft to ha...
1,Red-headed Tetra - Petitella (Hemigrammus) ble...,soft to medium hard,Tetra,,25-30 °C,top,,,100 l (ca. 80cm),hard,...,omnivorous - omnivorous,yes,yes,yes,4-8 cm,2 - normal,,,,Red-headed Tetra - Petitella (Hemigrammus) ble...
2,Blue Neon - Paracheirodon simulans,soft to medium hard,Tetra,,25-30 °C,center,,,54 l (ca. 60cm),hard,...,omnivorous - omnivorous,yes,yes,yes,1-4 cm,1 - easy,,,,Blue Neon - Paracheirodon simulans soft to med...
3,Red Neon XL - Paracheirodon axelrodi,soft to medium hard,Tetra,,25-30 °C,center,,,54 l (ca. 60cm),hard,...,omnivorous - omnivorous,yes,yes,yes,4-8 cm,1 - easy,,,,Red Neon XL - Paracheirodon axelrodi soft to m...
4,Neon Tetra - Paracheirodon innesi,soft to medium hard,Tetra,,25-30 °C,top,,,54 l (ca. 60cm),hard,...,omnivorous - omnivorous,yes,yes,yes,1-4 cm,2 - normal,,,,Neon Tetra - Paracheirodon innesi soft to medi...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,Dwarf Snakehead - Channa gachua blue bengal - ...,medium hard water,Other fish,,25-30 °C,below,,,250/300 l (ca. 120cm),medium,...,carnivore - meat eater,yes,no,no,> 12cm,4 - special,,,,Dwarf Snakehead - Channa gachua blue bengal - ...
539,Beautiful Snakehead - Channa pulchra - DNZ - 10cm,medium hard water,Other fish,,20-25 °C,below,,,250/300 l (ca. 120cm),,...,carnivore - meat eater,yes,no,no,> 12cm,4 - special,,,,Beautiful Snakehead - Channa pulchra - DNZ - 1...
540,Axolotl - Ambystoma mexicanum - DNZ,medium hard to hard,Other fish,,15-20 °C,below,,,100 l (ca. 80cm),simply,...,carnivore - meat eater,conditional (see description),conditional (see description),no,> 12cm,4 - special,,,,Axolotl - Ambystoma mexicanum - DNZ medium har...
541,Brown White-fronted Knifefish - Apteronotus al...,soft to hard,Other fish,,25-30 °C,center,,,600 l (ca. 180cm),hard,...,carnivore - meat eater,yes,no,no,> 12cm,4 - special,,,,Brown White-fronted Knifefish - Apteronotus al...


In [14]:
# check first element
df.text[0]

'Cardinalfish - Tanichthys albonubes soft to hard Beardies no 18-22 °C everywhere active yes, with peaceful fish 54 l (ca. 60cm) medium swarm behaviour with dwarf shrimp, offspring may be eaten Asia omnivorous - omnivorous yes yes yes 4-8 cm 1 - easy dynamic group behaviour no 3-5 years'

## Text vectorization

**Different tools** are available to **vectorize texts**. Dependent on your problem, you can use **NLP methods** like **CountVectorizer**, **TfidfVectorizor** or **Words2Vec**. In this example we will use the **TfidfVectorizer**. Documentation can be found [here](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html). **TF-IDF** or **Term Frequency Inverse Document Frequency** is a **statistical measure** to determine the **significance of different words** in a document. Subsequently every word gets a **TF-IDF value**. This value is the product of the **Term frequency (TF)** and the **inverse document frequency (IDF)**.

Let's have a look at an example with 3 sentences where each sentence represents a document:

1. **Recommender** are cool
2. **Recommender** are helpful
3. **Recommender** use your data

In these 3 sentences the term "**recommender**" occurs 3 times, the term "**are**" 2 times and "**use**" 1 time.

Let's calculate the **TF (term-frequency)** for those sentences.

1. [0.33, 0.33, 0.33] 
2. [0.33, 0.33, 0.33] 
3. [0.25, 0.25, 0.25, 0.25]

In sentence 1. are 3 unique words, the frequency of each word is therefore 1/3, same in sentence 2. Sentence 3 consists of 4 unique words with a corresponding frequency of 1/4.

Now, let's have a look at the **IDF (inverse document frequency)**

The **IDF** is the logarithm of the ratio of the **total number of documents** to the **number of documents** having this term. 

* The word "**recommender**" appears in all 3 documents and has therefore a **Log(3/3) = 0**
* The word "**are**" appears in two documents with a corresponding **Log(3/2) = 0.176**
* The words "**cool**", "**helpful**", etc appear in only one document leading to a **Log(3/1) = 0.477**

The **IDF** is a measure of how **much information** the **word** provides and as you can see, the **IDF** is **higher** the **rarer** the **word** is. This means **rare words** are **more meaningful to get the context of a document**.</br></br>


The TF-IDF of a word then is the product of the TF and the IDF value. The higher the score, the more relevant is this word for that particular document and with regards to recommender system the more important they are to distinguish categories.>

Now that we've learned how TF-IDF works, let's vectorize our data frame:

In [15]:
# vectorize text column
tf = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=0.01)
tfidf_matrix = tf.fit_transform(df["text"])

Since we now have a vecorized matrix of our data, we can likewise above calculate the cosine similarity on it.

In [16]:
# calculate cosine similarity
cosine_sim = linear_kernel(
    tfidf_matrix, tfidf_matrix
)  # linear_kernel is a useful function to calculate cosine sim on large data sets

And use the cosine similarity matrix to get our top 10 recommendations.

In [17]:
# Build index with fish names
names = df["name"]
indices = pd.Series(df.index, index=df["name"])


# Function that get fish recommendations based on the cosine similarity
def fish_recommendations(name, top_n=10):
    # get the index of the fish we put into the function
    idx = indices[name]
    # calculate all cosine similarities to that fish and store it in a list
    sim_scores = list(enumerate(cosine_sim[idx]))
    # sort the list staring with the highest similarity
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # get the similarities from 1:11 (not starting with 0 because it is the same fish)
    sim_scores = sim_scores[1 : top_n + 1]
    # get the indices of those 10 fish
    fish_indices = [i[0] for i in sim_scores]
    # return the fish names of that 10 fish
    return names.iloc[fish_indices]

In [20]:
# get recommendations
fish_recommendations("Blue Neon - Paracheirodon simulans", top_n=5)

128       Red Neon - Paracheirodon axelrodi - DNZ
3            Red Neon XL - Paracheirodon axelrodi
4               Neon Tetra - Paracheirodon innesi
69     Red-gold neon tetra - Paracheirodon innesi
123                Blue Peru Tetra - Knodus borki
Name: name, dtype: str

### Reading the results

For **Blue Neon - Paracheirodon simulans**, the TF-IDF recommender returns the Red Neon, Red Neon XL, Neon Tetra, Red-gold neon tetra, and Blue Peru Tetra. These are much closer matches than the feature-only recommender produced: several are close relatives from the same Paracheirodon genus. Weighting distinctive words (such as the species name) more heavily lets the text-based model pick up fine-grained similarity that equally-weighted one-hot features cannot.

Input:

Blue neon             |  
:-------------------------:|
<img src="https://blog.tetra.net/de-de/wp-content/uploads/2019/05/Platinum-Green-Neon-Fish-Paracheirodon-simulans_shutterstock_630636488_bearb-1200x800.jpg" alt="drawing" width="100"/> |

Pictures of Top 5 recommendations:

Red Neon            |  Red Neon XL           |  Neon Tetra          |  Blue Peru Tetra         |  Red-gold Neon   |      
:-------------------------:|:-------------------------: |:-------------------------: |:-------------------------: |:-------------------------:
<img src="https://www.garnelio.de/media/image/a3/36/17/Paracheirodon-axelrodi-roter-neon2VaNjbtTw3umpb.jpg" alt="drawing" width="100"/>  |  <img src="https://www.garnelio.de/media/image/a3/36/17/Paracheirodon-axelrodi-roter-neon2VaNjbtTw3umpb.jpg" alt="drawing" width="100"/>   |  <img src="https://www.aquarium-ratgeber.com/wp-content/uploads/2021/08/neonsalmler-paracheirodon-innesi.jpg" alt="drawing" width="100"/>  |  <img src="https://aquainfo.org/wp-content/uploads/2019/10/Knodus-borki.webp" alt="drawing" width="100"/> |  <img src="https://www.garnelio.de/media/image/b3/37/91/Garnelio-Zierfisch-neon-rot-gold-paracheirodon-axelrodi_600x600.jpg" alt="drawing" width="100"/> 

Inspecting the outcome, you can see that the most similar fish to the blue neon is the red neon, a close relative. If you want you can now play around with different fish names or use this notebook as a template for your own recommender :)

## Summary

In this notebook you:

- Built a content-based recommender that ranks fish by the cosine similarity of their features.
- Computed cosine similarity on one-hot encoded feature vectors (93 features across 543 fish).
- Applied TF-IDF to turn each fish's features into a text vector and built a text-based recommender.
- Generated and compared the top-N recommendations from both approaches, and saw that the TF-IDF model returned closer matches (neon relatives) for the Blue Neon.

## Bonus
If you want to learn how to build a more advanced text-based recommender system using the Word2Vec method, check out this [article](https://www.analyticsvidhya.com/blog/2019/07/how-to-build-recommendation-system-word2vec-python/).

## References & Further Reading

- [**scikit-learn: TfidfVectorizer**](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html): Reference for the TF-IDF vectoriser used to build the text-based recommender.
- [**scikit-learn: cosine_similarity**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html): Reference for the similarity metric used throughout this notebook.
- [**Google: Content-based filtering**](https://developers.google.com/machine-learning/recommendation/content-based/basics): A short, intuitive explanation of how content-based recommenders use item features.